## Example 1: jedi Script autocompletion functionality test 1

In [ ]:
import jedi

# Example 1: Code autocompletion with jedi.Script
print("Example 1: Python code autocompletion")

# Sample Python code with incomplete expressions
code_samples = [
    "import os\nos.pa",
    "import sys\nsys.ve", 
    "from datetime import datetime\ndatetime.no",
    "[1, 2, 3].ap",
    "{'key': 'value'}.ke"
]

print("Testing autocompletion:")
for i, code in enumerate(code_samples, 1):
    try:
        script = jedi.Script(code=code)
        completions = script.complete()
        
        print(f"  Sample {i}: '{code.split()[-1]}'")
        print(f"    Found {len(completions)} completions:")
        
        for comp in completions[:3]:  # Show first 3 completions
            print(f"      - {comp.name} ({comp.complete})")
            
    except Exception as e:
        print(f"  Sample {i}: Error - {e}")
        raise
    
    print()

print("Code autocompletion completed successfully")

## Example 2: jedi Script definitions functionality test 2

In [ ]:
import jedi

# Example 2: Finding definitions with jedi.Script.infer() (the replacement for the removed goto_definitions())
print("Example 2: Code definition lookup")

# Code with various identifiers to look up
code = """
import os
import sys
from datetime import datetime

def my_function():
    return "Hello"

class MyClass:
    def method(self):
        pass

variable = 42
result = my_function()
"""

script = jedi.Script(code=code)

# Test infer() for different elements: (line, column, description, expected (name, type) of the first inference)
test_positions = [
    (2, 7, "os", ("os", "module")),
    (3, 7, "sys", ("sys", "module")), 
    (6, 4, "my_function", ("my_function", "function")),
    (9, 6, "MyClass", ("MyClass", "class")),
    (13, 0, "variable", ("int", "instance")),  # a variable infers to the type of its value
    (14, 9, "my_function call", ("my_function", "function"))
]

print("Testing definition lookup:")
for line, column, description, expected in test_positions:
    try:
        definitions = script.infer(line=line, column=column)
        print(f"  {description}:")
        
        if definitions:
            for defn in definitions[:2]:  # Show first 2 definitions
                print(f"    - {defn.name} ({defn.type}) in {defn.module_name}")
        else:
            print(f"    No definitions found")
        assert definitions and (definitions[0].name, definitions[0].type) == expected, (description, definitions)
            
    except Exception as e:
        print(f"  {description}: Error - {e}")
        raise

print("Definition lookup completed successfully")

## Example 3: jedi Interpreter functionality test 3

In [ ]:
import jedi
import sys
import os

# Example 3: Interactive completion with jedi.Interpreter
print("Example 3: Interactive Python completion")

# Create runtime namespaces with actual objects
runtime_namespace = {
    'sys': sys,
    'os': os,
    'my_list': [1, 2, 3, 4, 5],
    'my_dict': {'name': 'Alice', 'age': 30, 'city': 'NYC'},
    'my_string': 'Hello World'
}

# Test completion with live objects
test_expressions = [
    "sys.ve",
    "os.pa", 
    "my_list.ap",
    "my_dict.ke",
    "my_string.up"
]

print("Testing interactive completion:")
for expr in test_expressions:
    try:
        # Create interpreter with current namespace
        interpreter = jedi.Interpreter(expr, [runtime_namespace])
        completions = interpreter.complete()
        
        print(f"  Expression: '{expr}'")
        print(f"    Found {len(completions)} completions:")
        
        for comp in completions[:3]:  # Show first 3
            print(f"      - {comp.name} -> {expr[:-2]}{comp.complete}")
            
    except Exception as e:
        print(f"  Expression '{expr}': Error - {e}")
        raise
    
    print()

print("Interactive completion completed successfully")

## Example 4: jedi Script names functionality test 4

In [ ]:
import jedi

# Example 4: Code analysis with jedi.Script.get_names()
print("Example 4: Python code name analysis")

# Sample Python code with various name types
code = """
import json
from collections import defaultdict

# Global variables
API_KEY = "secret_key"
config = {"debug": True}

# Function definitions
def process_data(data):
    result = []
    for item in data:
        result.append(item.upper())
    return result

# Class definitions
class DataProcessor:
    def __init__(self, name):
        self.name = name
        self.count = 0
    
    def process(self, items):
        self.count += len(items)
        return [item * 2 for item in items]

# Usage
processor = DataProcessor("main")
output = process_data(["hello", "world"])
"""

script = jedi.Script(code=code)

# Get all names in the code
print("Analyzing code names:")
names = script.get_names()

# Categorize names by type
name_categories = {}
for name in names:
    name_type = name.type
    if name_type not in name_categories:
        name_categories[name_type] = []
    name_categories[name_type].append(name.name)

# Display results
for category, name_list in name_categories.items():
    print(f"  {category}: {', '.join(sorted(set(name_list)))}")

print(f"\nTotal names found: {len(names)}")
print("Code name analysis completed successfully")

## Example 5: jedi Script help functionality test 5

In [ ]:
import jedi

# Example 5: Function documentation with jedi.Script.help()
print("Example 5: Python function help and documentation")

# Code with function calls to get help for
code_samples = [
    ("len", "len("),
    ("print", "print("),
    ("range", "range("),
    ("open", "open("),
    ("sorted", "sorted(")
]

print("Getting help documentation:")
for func_name, code in code_samples:
    try:
        # Create script and position cursor after function name
        script = jedi.Script(code=code)
        help_info = script.help()
        
        print(f"  Function: {func_name}")
        
        if help_info:
            for info in help_info[:1]:  # Show first help entry
                # Get the description or docstring
                if hasattr(info, 'docstring'):
                    doc = info.docstring()
                    # Show first line of docstring
                    first_line = doc.split('\n')[0] if doc else "No documentation"
                    print(f"    Documentation: {first_line}")
                else:
                    print(f"    Type: {info.name}")
        else:
            print(f"    No help found")
            
    except Exception as e:
        print(f"  {func_name}: Error - {e}")
        raise
    
    print()

# Test with custom function
custom_code = '''
def calculate_area(length, width):
    """Calculate the area of a rectangle.
    
    Args:
        length: The length of the rectangle
        width: The width of the rectangle
    
    Returns:
        The area as length * width
    """
    return length * width

calculate_area(
'''

try:
    script = jedi.Script(code=custom_code)
    help_info = script.help()
    if help_info:
        doc = help_info[0].docstring()
        print(f"Custom function help: {doc.split('.')[0]}.")
except:
    print("Custom function help not available")
    raise

print("Function help documentation completed successfully")